# BOC predictions per treatment group and timepoint — MM-ViT16 (run 1)

| Row | Timepoint | Patients | Images | Model input |
|---|---|---|---|---|
| Experimental (ketotifen) | Baseline | P1, P2, P4, P6, P10 | `Export/experimental_basline_crop/p<N>/p<N>_{1,2,3}.tif` | **image + baseline kPa (rescaled)** (+ image only for reference) |
| Experimental | 1 week of ketotifen | P1, P2, P4, P6, P10 | `02_data/artifact source/swe_patient_BOC/` (`_new` if present) | image + kPa (as in the earlier analysis) |
| Experimental | Mid-treatment | P1, P2, P4, P6, P10 | `Export/new_clustering/mid/p<N>/` | image only |
| Control | Baseline | P11–P15 | `02_data/artifact source/swe_patient_BOC/` | image + kPa (as in the earlier analysis) |
| Control | Mid-treatment | P11–P15 | `Export/new_clustering/mid/p<N>/`; **P11 + 2 extra frames** from `Export/experimental_basline_crop/p11_mid_extra Imgaes/` | image only |

**Input rule:** image + elastic modulus (rescaled) for baseline / early timepoints; image only for mid-treatment.
The image-only baseline result for the Experimental group (1/5) is kept in `summary_group_timepoint.csv` as a reference.

**Rules fixed before running:** patient prediction = mean probability over **all** of that patient's images at that timepoint
(no frame selection; P11 mid now uses all 5 frames, replacing the earlier post-hoc single frame).
Labels: RECIST 1.1 from `02_data/artifact source/NEW_DATA.xlsx`. kPa rescaling: label-free distribution matching
(clinical cohort → murine training scale), identical to the baseline notebook.
The earlier image + kPa panels (Experimental 1 week, Control baseline) used the **post-ketotifen / final** stiffness from
`clinical table.xlsx`; this is reproduced unchanged and noted in the figure legend.

In [ ]:
import json, os, re, tempfile, zipfile
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf
from PIL import Image
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import confusion_matrix

# locate prognostic-ml root (folder containing 03_models) from the notebook location
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '03_models').is_dir())
EXPORT = ROOT.parent / 'Export'
OUT = Path.cwd()                      # results + figures are written next to this notebook
FIG = OUT / 'figures'; FIG.mkdir(exist_ok=True)
SRC = ROOT / '02_data' / 'artifact source'
CLASS_NAMES = ['response', 'stable', 'non-response']
print('ROOT  :', ROOT); print('EXPORT:', EXPORT); print('OUT   :', OUT); print('TF', tf.__version__)

## 1. Model loading

In [ ]:
class TokenSlice(layers.Layer):
    def __init__(self, index=0, **kw): super().__init__(**kw); self.index = index
    def call(self, x): return x[:, self.index, :]
    def get_config(self): return {**super().get_config(), 'index': self.index}

class CLSToken(layers.Layer):
    def __init__(self, dim, **kw): super().__init__(**kw); self.dim = dim
    def build(self, s): self.cls = self.add_weight(name='cls', shape=(1, 1, self.dim), initializer='random_normal')
    def call(self, x): return tf.concat([tf.tile(self.cls, [tf.shape(x)[0], 1, 1]), x], axis=1)
    def get_config(self): return {**super().get_config(), 'dim': self.dim}

class AddConstant(layers.Layer):
    def __init__(self, value, **kw):
        super().__init__(**kw); self.value = np.asarray(value, np.float32); self._c = tf.constant(self.value)
    def call(self, x): return x + self._c
    def get_config(self): return {**super().get_config(), 'value': self.value.tolist()}

CUSTOM = {'TokenSlice': TokenSlice, 'CLSToken': CLSToken, 'AddConstant': AddConstant}

def _patch(o):
    if isinstance(o, dict):
        if o.get('class_name') == 'Lambda' and o['config']['name'] == 'cls_pick':
            o.update(class_name='TokenSlice', module=None, registered_name='TokenSlice',
                     config={'name': 'cls_pick', 'trainable': True, 'dtype': 'float32', 'index': 0})
        if o.get('class_name') == 'Add' and '_CONSTANT_VALUE' in json.dumps(o.get('inbound_nodes', []))[:200]:
            node = o['inbound_nodes'][0]
            t = [e for e in node if e[0] != '_CONSTANT_VALUE']; c = [e for e in node if e[0] == '_CONSTANT_VALUE']
            o.update(class_name='AddConstant', module=None, registered_name='AddConstant',
                     config={'name': o['config']['name'], 'trainable': True, 'dtype': 'float32', 'value': c[0][2]},
                     inbound_nodes=[t]); o.pop('build_config', None)
        for v in o.values(): _patch(v)
    elif isinstance(o, list):
        for v in o: _patch(v)

def load_keras(path):
    with zipfile.ZipFile(path) as z:
        cfg = json.loads(z.read('config.json')); _patch(cfg)
        tmp = tempfile.NamedTemporaryFile(suffix='.keras', delete=False); tmp.close()
        with zipfile.ZipFile(tmp.name, 'w') as o:
            for n in z.namelist(): o.writestr(n, json.dumps(cfg) if n == 'config.json' else z.read(n))
    m = tf.keras.models.load_model(tmp.name, custom_objects=CUSTOM, compile=False, safe_mode=False)
    os.unlink(tmp.name); return m

MODEL_MM = ROOT / '03_models/MM-ViT16/run1/best_model_1.keras'
MODEL_IO = ROOT / '03_models/MM-ViT16/run1/best_model_image_only_1.keras'
mm, io = load_keras(MODEL_MM), load_keras(MODEL_IO)
cfg = json.load(open(ROOT / '04_artifacts/MM-ViT16/run1/model_config.json'))
print(f'multimodal params {mm.count_params():,}  (expected 315,011)')
print(f'image-only params {io.count_params():,}  (expected 314,883)')
print('model kPa standardisation: mean %.3f  SD %.3f' % (cfg['scaler_mean'], cfg['scaler_std']))

## 2. Labels, groups, stiffness and rescaling

In [ ]:
lab = pd.read_excel(SRC / 'NEW_DATA.xlsx', header=None, names=['study_id', 'recist', 'x'])
lab['pn'] = lab.study_id.str.extract(r'0*(\d+)')[0].astype(int)
TRUE = dict(zip(lab.pn, lab.recist.map({'PR': 'response', 'SD': 'stable', 'PD': 'non-response'})))
EXP, CTRL = [1, 2, 4, 6, 10], [11, 12, 13, 14, 15]

ct = pd.read_excel(EXPORT / 'SWE BOC/clinical table.xlsx', header=None).iloc[24:39, [0, 5, 6]]
ct.columns = ['p', 'kpa_baseline', 'kpa_post']; ct['pn'] = ct.p.str.extract(r'(\d+)')[0].astype(int)
KPA_BASE = dict(zip(ct.pn, ct.kpa_baseline.astype(float))); KPA_POST = dict(zip(ct.pn, ct.kpa_post.astype(float)))

raw = pd.read_csv(EXPORT / 'Original_excel_results ketotifen - April 2026 - FINAL.xlsx - data.csv', header=None, dtype=str)
clin, pid = [], None
for _, r in raw.iterrows():
    m = re.match(r'0*(\d+)-Ket', str(r[0]).strip()); pid = int(m.group(1)) if m else pid
    d = pd.to_datetime(r[6], format='%m/%d/%Y', errors='coerce')
    v = [float(x) for x in re.findall(r'\d+\.?\d*', str(r[7]).replace('25.1.', '25.1,'))] if pd.notna(r[7]) else []
    if pid and pd.notna(d) and v: clin.append(np.mean(v))
clin = np.array(clin)
mouse = pd.read_excel(ROOT / '02_data/clustering_all_v5.xlsx')['Elastic Modulus SWE (kPa)'].to_numpy(float)
rescale = lambda k: mouse.mean() + mouse.std() * (np.asarray(k, float) - clin.mean()) / clin.std()
z = lambda k: ((np.asarray(k, float) - cfg['scaler_mean']) / cfg['scaler_std']).reshape(-1, 1).astype(np.float32)
print(f'rescaling: kPa_rescaled = {mouse.mean() - mouse.std()/clin.std()*clin.mean():.2f} + {mouse.std()/clin.std():.3f} x kPa')

## 3. Image loaders (identical to the earlier notebooks)

In [ ]:
PCOLS = ['p_response', 'p_stable', 'p_non_response']
def load_tif(p):     # tumour-segmented 400x300 canvas -> white background -> 304x400 -> /255
    x = np.asarray(Image.open(p).convert('RGBA'))
    rgb = np.where(x[..., 3:] > 0, x[..., :3], 255).astype(np.float32)
    if rgb.shape[:2] != (300, 400): rgb = np.asarray(Image.fromarray(rgb.astype(np.uint8)).resize((400, 300), Image.BILINEAR), np.float32)
    return tf.clip_by_value(tf.image.resize(rgb, [304, 400]) / 255.0, 0, 1).numpy()
def load_png(p):     # earlier BOC PNGs: decode -> 304x400 -> /255
    x = tf.image.decode_png(tf.io.read_file(str(p)), channels=3)
    return tf.clip_by_value(tf.image.resize(x, [304, 400]) / 255.0, 0, 1).numpy()

def run(rows, model, kpa=None):
    """rows: DataFrame with pn, path, loader. Returns per-image + per-patient (mean probability) tables."""
    X = np.stack([f(p) for f, p in zip(rows.loader, rows.path)])
    prob = model.predict(X if kpa is None else [X, z(kpa)], verbose=0)
    im = rows.drop(columns='loader').copy(); im[PCOLS] = prob
    im['pred'] = np.array(CLASS_NAMES)[prob.argmax(1)]; im['true'] = im.pn.map(TRUE); im['correct'] = im.pred == im.true
    g = im.groupby('pn')[PCOLS].mean().reset_index(); g['n_images'] = im.groupby('pn').size().values
    g['pred'] = np.array(CLASS_NAMES)[g[PCOLS].to_numpy().argmax(1)]; g['true'] = g.pn.map(TRUE); g['correct'] = g.pred == g.true
    return im, g

def frame(paths, loader):
    d = pd.DataFrame({'path': [Path(p) for p in paths]})
    d['pn'] = d.path.map(lambda p: int(re.match(r'p(\d+)', p.name).group(1))); d['image'] = d.path.map(lambda p: p.name)
    d['loader'] = loader; return d.sort_values(['pn', 'image']).reset_index(drop=True)

## 4. Predictions

In [ ]:
R = {}   # (group, timepoint) -> per-patient table used in the figure
IMGS = {}

# Experimental · baseline (NEW): image + baseline kPa rescaled (figure) + image only (reference)
eb = frame(sorted(p for p in (EXPORT / 'experimental_basline_crop').glob('p*/p*_[0-9].tif') if re.fullmatch(r'p\d+', p.parent.name)), load_tif)
assert sorted(eb.pn.unique()) == EXP and len(eb) == 15
IMGS['exp_base_mm'], R[('Experimental', 'Baseline')] = run(eb, mm, rescale(eb.pn.map(KPA_BASE)))   # figure: image + baseline kPa (rescaled)
IMGS['exp_base_io'], exp_base_io = run(eb, io)                                       # reference: image only

# Experimental 1 week & Control baseline: earlier swe_patient_BOC images, image + post-ketotifen kPa (as before)
IMG = SRC / 'swe_patient_BOC'
pick = [IMG / (f'p{n}_new.png' if (IMG / f'p{n}_new.png').exists() else f'p{n}.png') for n in EXP + CTRL]
sb = frame(pick, load_png)
IMGS['boc'], boc = run(sb, mm, rescale(sb.pn.map(KPA_POST)))
R[('Experimental', '1 week ketotifen')] = boc[boc.pn.isin(EXP)].reset_index(drop=True)
R[('Control', 'Baseline')] = boc[boc.pn.isin(CTRL)].reset_index(drop=True)

# Mid-treatment: image only, all frames; P11 = 3 frames + 2 extra frames
mp = sorted((EXPORT / 'new_clustering/mid').glob('p*/p*_[0-9].tif')) + sorted((EXPORT / 'experimental_basline_crop').glob('p11_mid_extra*/*.tif'))
md_ = frame(mp, load_tif)
assert md_.groupby('pn').size().to_dict()[11] == 5
IMGS['mid'], mid = run(md_, io)
R[('Experimental', 'Mid-treatment')] = mid[mid.pn.isin(EXP)].reset_index(drop=True)
R[('Control', 'Mid-treatment')] = mid[mid.pn.isin(CTRL)].reset_index(drop=True)

summary = pd.DataFrame([dict(group=g, timepoint=t, correct=int(d.correct.sum()), n=len(d),
                             wrong=', '.join(f'P{n}' for n in d.loc[~d.correct, 'pn'])) for (g, t), d in R.items()])
summary.loc[len(summary)] = dict(group='Experimental', timepoint='Baseline [reference: image only]',
                                 correct=int(exp_base_io.correct.sum()), n=5,
                                 wrong=', '.join(f'P{n}' for n in exp_base_io.loc[~exp_base_io.correct, 'pn']))
pd.concat([d.assign(group=g, timepoint=t) for (g, t), d in R.items()]).to_csv(OUT / 'per_patient_group_timepoint.csv', index=False)
exp_base_io.assign(group='Experimental', timepoint='Baseline (image only, reference)').to_csv(OUT / 'per_patient_exp_baseline_image_only.csv', index=False)
pd.concat([d.assign(set=k) for k, d in IMGS.items()]).to_csv(OUT / 'per_image_all.csv', index=False)
summary.to_csv(OUT / 'summary_group_timepoint.csv', index=False)
summary

In [ ]:
# patient-level detail
show = pd.concat([d.assign(group=g, timepoint=t) for (g, t), d in R.items()])
show['probs PR/SD/PD'] = show[PCOLS].round(2).astype(str).agg(' / '.join, axis=1)
show[['group', 'timepoint', 'pn', 'n_images', 'true', 'pred', 'correct', 'probs PR/SD/PD']]

## 5. Figure — one row per treatment group, one confusion matrix per timepoint

In [ ]:
MM_ = 1 / 25.4
plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.labelsize': 7, 'axes.titlesize': 7,
    'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'axes.linewidth': 0.5, 'xtick.major.width': 0.5,
    'ytick.major.width': 0.5, 'xtick.major.size': 2, 'ytick.major.size': 2, 'pdf.fonttype': 42, 'ps.fonttype': 42})
AB = {'response': 'PR', 'stable': 'SD', 'non-response': 'PD'}
CMAP = LinearSegmentedColormap.from_list('pastel', ['#ffffff', '#9CC5E3', '#2c5d8f'])
INPUT = {'Baseline': {'Experimental': 'image + kPa', 'Control': 'image + kPa'},
         '1 week ketotifen': {'Experimental': 'image + kPa'}, 'Mid-treatment': {'Experimental': 'image only', 'Control': 'image only'}}

def cm_panel(ax, d, title, sub):
    cm = confusion_matrix(d.true, d.pred, labels=CLASS_NAMES)
    ax.imshow(cm, cmap=CMAP, vmin=0, vmax=5)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8, color='white' if cm[i, j] >= 3 else 'black')
    ax.set_xticks(range(3), [AB[c] for c in CLASS_NAMES]); ax.set_yticks(range(3), [AB[c] for c in CLASS_NAMES])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True (RECIST 1.1)')
    ok = int(d.correct.sum())
    ax.set_title(f'{title}\n{sub} · {ok}/{len(d)} correct', fontsize=7)

cols = ['Baseline', '1 week ketotifen', 'Mid-treatment']
fig, axes = plt.subplots(2, 3, figsize=(150 * MM_, 105 * MM_))
for r, grp in enumerate(['Experimental', 'Control']):
    for c, tp in enumerate(cols):
        ax = axes[r, c]
        if (grp, tp) in R:
            cm_panel(ax, R[(grp, tp)], tp, INPUT[tp][grp])
        else:
            ax.axis('off'); ax.text(0.5, 0.5, 'not applicable\n(no ketotifen)', ha='center', va='center', fontsize=6.5, color='0.5', transform=ax.transAxes)
    axes[r, 0].annotate(f'{grp}\n(P{", P".join(map(str, EXP if grp == "Experimental" else CTRL))})', xy=(-0.62, 0.5),
                        xycoords='axes fraction', ha='center', va='center', rotation=90, fontsize=7.5, fontweight='bold')
fig.tight_layout(w_pad=2.5, h_pad=2.0)
fig.savefig(FIG / 'Figure_confusion_by_group_timepoint.pdf', bbox_inches='tight')
fig.savefig(FIG / 'Figure_confusion_by_group_timepoint.png', dpi=600, bbox_inches='tight')
plt.show(); print('Saved to', FIG)